In [2]:
import numpy as np
import pandas as pd 

/Users/rf50/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [3]:
df_orig = pd.read_excel('UChicago Carillon Library Copy.xlsx')

df_orig = df_orig.dropna(subset=['Title'])

/Users/rf50/opt/anaconda3/lib/python3.9/site-packages/openpyxl/worksheet/_read_only.py:85: UserWarning: Unknown extension is not supported and will be removed
  for idx, row in parser.parse():


In [4]:
df_tag = pd.read_csv("composers_tagged.csv")
df_tag['style_full'] = df_tag['style'] + ' - ' + df_tag['style_sub']

In [5]:
df_clean = df_orig[[
    'Composer', 'Arranger', 'Title',
    'Book/Volume?', 'Pages', 'Duet?']
].rename(
    columns={'Composer':'composer', 'Arranger':'arranger', 'Title':'title',
        'Book/Volume?':'book', 'Pages':'pages', 'Duet?':'duet'})

df_clean = df_clean.merge(df_tag, how='left', on='composer')

df_clean['search_string'] = df_clean['title'].astype(str) + ' by ' + df_clean['composer'].astype(str)

In [6]:
df_clean.columns

Index(['composer', 'arranger', 'title', 'book', 'pages', 'duet', 'style',
       'style_sub', 'style_full', 'region', 'region_sub', 'confidence',
       'search_string'],
      dtype='object')

### Back out style_intermediate

In [54]:
def simplify_style(value):
    value_lower = str(value).lower()

    if "unknown" in value_lower:
        return value
    elif "late romantic" in value_lower:
        return "Classical - Romantic"
    elif "classical" in value_lower:
        return value
    elif "popular" in value_lower:
        return value
    elif "hymn" in value_lower:
        return "Hymn"
    elif "traditional" in value_lower:
        return "Traditional"
    else:
        return value

df_clean["style_intermediate"] = df_clean["style_full"].apply(simplify_style)

### Validation

In [11]:
# 301 low-confidence tags by composer - can do this piece 
len(df_clean.loc[df_clean['confidence']=='Low', 'composer'].unique())

301

In [19]:
df_clean.loc[df_clean['confidence']=='Low'
    ].groupby('composer').size().reset_index(name='count'
    ).sort_values(by='count', ascending=False).iloc[:50]

,composer,count
22,"Berghuis, Johannes",50
30,"Borghuis, Karel",28
110,"Hummel, A.",23
244,"Uten, Eugeen",21
116,"Johnson, Alex",20
245,"Uten, Eugen",19
270,"Whipple, Laura Hewitt",17
90,"Gordon, John Douglas",10
39,"Butler, Joseph",9
280,"Zettek, Charles Jr.",8


In [56]:
df_clean['style_intermediate'].value_counts()

style_intermediate
Classical - Contemporary Classical    908
Classical - Modern Classical          739
Classical - Baroque                   665
Traditional                           615
Classical - Romantic                  609
Hymn                                  403
Unknown - Unknown                     368
Classical - Classical                 299
Popular - Pop                         150
Popular - Film                         97
Classical - Early Music                92
Popular - Musical                      65
Popular - Jazz                         53
Classical - Impressionist              35
Classical - Unknown                     8
Name: count, dtype: int64

### Adding written-for-carillon tag?

In [42]:
df_orig['Publisher'].value_counts()[:20]

Publisher
GCNA                             396
ACME                             178
Yale                              67
ACW                               43
Mechelen                          38
NKV                               21
CGNA                              20
NACS                              19
Alice Gomez                       17
Scherzando                        17
NKV (Donemus)                     14
René Vanstreels                    9
Mayo Clinic                        9
Imprimerie St-Joseph               8
Sheet Music Plus                   7
Margo Halsted                      7
Donemus                            6
ACA                                6
The Curtis Institute of Music      4
UMich                              4
Name: count, dtype: int64

In [51]:
df_orig.loc[~pd.isna(df_orig['Publisher']),
    ].merge(df_tag, how='left', left_on='Composer', right_on='composer')[[
        'Composer', 'Arranger', 'Title', "Book/Volume?", 'Publisher', 'Year', 
        'style', 'style_sub', 'region', 'region_sub', 'confidence'
    ]]['style'].value_counts()

style
Classical      821
Unknown         56
Traditional     54
Popular         26
Hymn            25
Name: count, dtype: int64

In [37]:
df_orig.loc[~pd.isna(df_orig['Publisher']),
    ['Composer', 'Arranger']].iloc[:40]


,Composer,Arranger,Title,Book/Volume?,Pages,Carillon Plus?,Duration,Missing?,Publisher,Year,Recording,Duet?,Original?,Photocopy?,Reg?,Occasion,Genre,Other Tag,Unnamed: 18
0,?,"Allard, Émilien",?,NaN,2,NaN,NaN,NaN,Imprimerie St-Joseph,1974,NaN,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,"Adam, Adolphe","van Ulft, Carlo",Andante,Springfield Collection: Volume 1,6,NaN,NaN,NaN,NACS,2019,NaN,NaN,Y,N,NaN,NaN,NaN,NaN,NaN
6,"Adam, Adolphe","Gerken, Albert",Cantique de Noël,NaN,2,NaN,NaN,NaN,ACME,1984,NaN,NaN,NaN,NaN,NaN,Christmas,NaN,NaN,NaN
7,"Adam, Adolphe","van Ulft, Carlo",Galop General,Springfield Collection: Volume 1,9,NaN,NaN,NaN,NACS,2019,NaN,NaN,Y,N,NaN,NaN,NaN,NaN,NaN
8,"Adam, Adolphe","van Ulft, Carlo",Pas de Deux,Springfield Collection: Volume 1,5,NaN,NaN,NaN,NACS,2019,NaN,NaN,Y,N,NaN,NaN,NaN,NaN,NaN
12,"Aharony, Iddo",NaN,...the way nets cannot hold water,NaN,5,NaN,NaN,NaN,ACME,2018,NaN,NaN,Y,NaN,NaN,Ear Taxi festival,NaN,NaN,NaN
19,"Albeniz, Isaac","Disler, Mary Jo",Sevilla,NaN,7,NaN,NaN,NaN,ACME,2017,NaN,NaN,Y,NaN,NaN,NaN,NaN,NaN,NaN
21,"Albéniz, Isaac","Gerken, Albert",Leyenda (Legend),NaN,6,NaN,NaN,NaN,ACME,1998,NaN,N,NaN,NaN,Y,NaN,NaN,NaN,NaN
24,"Albinoni, Tomaso","Barnes, Ronald",Concerto Grosso,NaN,13,NaN,NaN,NaN,GCNA,2005,NaN,Y,NaN,NaN,NaN,NaN,NaN,NaN,NaN
29,"Allard, Émilien",NaN,Arabesque,NaN,5,NaN,"4'15""",NaN,GCNA,1991,we have another edition of this in the gouwens...,N,NaN,Y,Y,NaN,NaN,NaN,NaN


In [39]:
df_orig.loc[~pd.isna(df_orig['Original?'])].iloc[:10]

,Composer,Arranger,Title,Book/Volume?,Pages,Carillon Plus?,Duration,Missing?,Publisher,Year,Recording,Duet?,Original?,Photocopy?,Reg?,Occasion,Genre,Other Tag,Unnamed: 18
2,?,"Hart, Leen't",Folk Songs?,NaN,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Y,N,NaN,NaN,NaN,NaN,NaN
5,"Adam, Adolphe","van Ulft, Carlo",Andante,Springfield Collection: Volume 1,6,NaN,NaN,NaN,NACS,2019,NaN,NaN,Y,N,NaN,NaN,NaN,NaN,NaN
7,"Adam, Adolphe","van Ulft, Carlo",Galop General,Springfield Collection: Volume 1,9,NaN,NaN,NaN,NACS,2019,NaN,NaN,Y,N,NaN,NaN,NaN,NaN,NaN
8,"Adam, Adolphe","van Ulft, Carlo",Pas de Deux,Springfield Collection: Volume 1,5,NaN,NaN,NaN,NACS,2019,NaN,NaN,Y,N,NaN,NaN,NaN,NaN,NaN
12,"Aharony, Iddo",NaN,...the way nets cannot hold water,NaN,5,NaN,NaN,NaN,ACME,2018,NaN,NaN,Y,NaN,NaN,Ear Taxi festival,NaN,NaN,NaN
19,"Albeniz, Isaac","Disler, Mary Jo",Sevilla,NaN,7,NaN,NaN,NaN,ACME,2017,NaN,NaN,Y,NaN,NaN,NaN,NaN,NaN,NaN
43,"Allard, Émilien",NaN,Image No. 2,NaN,4,NaN,"5'15""",NaN,GCNA,1987,NaN,N,Y,Y,Y,NaN,NaN,NaN,NaN
44,"Allard, Émilien",NaN,Image No. 3,NaN,3,NaN,"4'45""",NaN,NaN,1960,NaN,N,N,NaN,NaN,NaN,NaN,bok,NaN
46,"Allard, Émilien",NaN,Imagerie No. 1,NaN,3,NaN,NaN,NaN,NaN,NaN,NaN,N,Y,N,NaN,NaN,NaN,NaN,NaN
53,"Allard, Émilien",NaN,Notule No. 1,NaN,3,NaN,NaN,NaN,GCNA,1987,NaN,N,Y,Y,Y,NaN,NaN,NaN,NaN


In [31]:
df_orig['Book/Volume'].value_counts()

,Composer,Arranger,Title,Book/Volume?,Pages,Carillon Plus?,Duration,Missing?,Publisher,Year,Recording,Duet?,Original?,Photocopy?,Reg?,Occasion,Genre,Other Tag,Unnamed: 18
0,?,"Allard, Émilien",?,NaN,2,NaN,NaN,NaN,Imprimerie St-Joseph,1974,NaN,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,?,"Allard, Émilien",Berceuse,NaN,2,NaN,NaN,Y,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,?,"Hart, Leen't",Folk Songs?,NaN,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Y,N,NaN,NaN,NaN,NaN,NaN
3,???,"Myhre, Milford",Washington's March No. 1,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Patriotic,NaN,NaN,NaN
4,17th Century Lullaby,"Hart, Leen't",Golden slumbers,Volksliederen II: volksliedjes met variaties u...,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5101,"Zinoviev, V. N.","Haazen, Jo",Evening Bells,Carillon Part IV,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5102,"Zipoli, Domenico","Lodine, Robert","Suite II in g minor (Preludio, Corrente, Sarab...",NaN,7,NaN,06:00:00,NaN,ACME,1986,NaN,NaN,Y,N,NaN,NaN,NaN,NaN,NaN
5103,"Zubel, Agata",NaN,Memory of Bronze,NaN,20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5104,"Zwart, Boudewijn",NaN,Giocoso Frescamente,NaN,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Save processed df

In [58]:
df_clean = df_clean[['composer', 'arranger', 'title', 'book', 'pages', 'duet', 'style',
       'style_sub', 'style_intermediate', 'style_full', 'region', 'region_sub', 'confidence', 'search_string']]

df_clean.to_csv('Processed_Carillon_Library.csv', index=False)

In [53]:
[
    'Hymn',
    'Traditional',
    'Classical - Early Music',
    'Classical - Baroque',
    'Classical - Classical',
    'Classical - Romantic',
    'Classical - Impressionist',
    'Classical - Modern Classical',
    'Classical - Contemporary Classical',
    'Classical - Unknown',
    'Popular - Jazz',
    'Popular - Musical',
    'Popular - Film',
    'Popular - Pop',
    'Unknown - Unknown',
]

array(['Unknown - Unknown', 'Classical - Baroque', 'Classical - Romantic',
       'Classical - Early Music', 'Classical - Contemporary Classical',
       'Popular - Pop', 'Classical - Modern Classical', 'Hymn',
       'Popular - Film', 'Classical - Classical', 'Popular - Musical',
       'Traditional', 'Classical - Unknown', 'Popular - Jazz',
       'Classical - Impressionist', 'Classical - Late Romantic'],
      dtype=object)